In [43]:
import os
import pickle
import torch
import pandas as pd
import torch.nn as nn
import re
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cpu


In [44]:
class LSTM_pt(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTM_pt, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.output_size = output_size

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=self.num_layers, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.linear = nn.Linear(hidden_size, output_size)
        self.activation = nn.LeakyReLU(negative_slope=0.01)

    def forward(self, inputs, hidden_state, cell_state):
        lstm_out, (h_n, c_n) = self.lstm(inputs, (hidden_state, cell_state))
        dropped_out = self.dropout(lstm_out)
        output = self.linear(dropped_out)
        output = self.activation(output)
        return h_n, c_n, output

In [45]:
def predict_house_price(model, row, label_encoders, scaler, device):

    # Make a copy of the row to avoid modifying the original
    row_copy = row.copy()

    # Convert row to DataFrame if it's a Series
    if isinstance(row_copy, pd.Series):
        row_copy = pd.DataFrame([row_copy])

    # Create a new DataFrame with only the columns we need to avoid issues
    processed_row = pd.DataFrame()

    # Handle the 'month' column before other processing if it's in YYYY-MM format
    if 'month' in row_copy.columns:
        # Check if it's a string in format like "2025-01"
        if isinstance(row_copy['month'].iloc[0], str) and re.match(r'^\d{4}-\d{2}$', row_copy['month'].iloc[0]):
            # Extract year and month as separate components
            month_str = row_copy['month'].iloc[0]
            parts = month_str.split('-')
            if len(parts) == 2:
                try:
                    year = int(parts[0])
                    month = int(parts[1])
                    # Convert to a more meaningful representation: months since 2000-01
                    # This creates a continuous timeline that better represents the progression of time
                    processed_row['month'] = (year - 2000) * 12 + month
                except:
                    processed_row['month'] = 0  # Default if parsing fails
            else:
                processed_row['month'] = 0  # Default for invalid format
        elif isinstance(row_copy['month'].iloc[0], pd.Timestamp):
            # If it's already a timestamp, convert to months since 2000-01
            timestamp = row_copy['month'].iloc[0]
            processed_row['month'] = (timestamp.year - 2000) * 12 + timestamp.month
        else:
            # Use the value as is if it's already processed
            processed_row['month'] = row_copy['month']
    else:
        # If month is missing, add a default value
        processed_row['month'] = 0

    # Step 1: Process categorical columns
    categorical_cols = ['town', 'flat_type', 'street_name',
                        'storey_range', 'flat_model', 'remaining_lease']

    for col in categorical_cols:
        if col in row_copy.columns and col in label_encoders:
            # Convert to string first to match the encoding process
            string_values = row_copy[col].astype(str)

            # Replace 'nan' strings with a default value
            string_values = string_values.replace('nan', 'Unknown')

            encoder = label_encoders[col]

            try:
                # Try to transform using the provided encoder
                processed_row[col] = encoder.transform(string_values)
            except ValueError as e:
                print(f"Found new category in {col}. Using default encoding.")

                # Instead of modifying the encoder (which could cause issues with other predictions),
                # we'll use a default value for unknown categories

                # Use the mean encoding value or a safe default like 0
                if len(encoder.classes_) > 0:
                    # Use the average of existing class indices as a reasonable default
                    default_value = np.mean(range(len(encoder.classes_)))
                    processed_row[col] = default_value
                else:
                    # If encoder is empty (unlikely), use 0
                    processed_row[col] = 0

                # Log the unknown values for reference
                unique_values = list(string_values.unique())
                current_classes = list(encoder.classes_)
                new_classes = [val for val in unique_values if val not in current_classes]
                print(f"New categories in {col} (using default encoding): {new_classes}")
        else:
            # If column is missing, add a default value
            processed_row[col] = 0

    # Step 2: Process continuous columns
    # Added 'month' to continuous columns so it's properly scaled
    continuous_cols = ['floor_area_sqm', 'lease_commence_date', 'month', 'resale_price']

    for col in continuous_cols:
        if col != 'resale_price' or col in row_copy.columns:  # Skip resale_price if not in input
            if col in row_copy.columns and col != 'month':  # Skip month since we already processed it
                # Handle potential NaN values
                if row_copy[col].isna().any():
                    row_copy[col] = row_copy[col].fillna(0)
                processed_row[col] = row_copy[col].astype(float)
            elif col == 'month':
                # We already processed month, so just ensure it's float type
                processed_row['month'] = processed_row['month'].astype(float)
            else:
                # If column is missing, add a default value
                processed_row[col] = 0.0

    # Fill any remaining NaN values
    processed_row = processed_row.fillna(0)

    # Step 3: Normalize the continuous columns
    # Get indices of continuous columns that are in the processed DataFrame
    cols_to_scale = [col for col in continuous_cols if col in processed_row.columns]
    if cols_to_scale:
        try:
            # Check if month column exists in scaler
            if 'month' in cols_to_scale and not hasattr(scaler, 'feature_names_in_') or 'month' not in scaler.feature_names_in_:
                # If month not in scaler, scale it separately
                month_col = processed_row['month'].copy()

                # Scale without month
                scaled_cols = [c for c in cols_to_scale if c != 'month']
                if scaled_cols:
                    processed_row[scaled_cols] = scaler.transform(processed_row[scaled_cols])

                # Add back month (normalized between 0-1 based on a reasonable range)
                # Assuming the earliest housing data might be from 2000 and we're in 2025 (300 months total range)
                max_month_value = 300  # 25 years * 12 months
                processed_row['month'] = month_col / max_month_value
            else:
                # Standard scaling with all columns
                processed_row[cols_to_scale] = scaler.transform(processed_row[cols_to_scale])
        except Exception as e:
            # print(f"Error during scaling: {e}")
            # Handle scaling error by manually scaling each column
            for col in cols_to_scale:
                try:
                    if hasattr(scaler, 'feature_names_in_') and col in scaler.feature_names_in_:
                        col_idx = list(scaler.feature_names_in_).index(col)
                        min_val = scaler.data_min_[col_idx]
                        max_val = scaler.data_max_[col_idx]
                        if max_val > min_val:
                            processed_row[col] = (processed_row[col] - min_val) / (max_val - min_val)
                        else:
                            processed_row[col] = 0
                    elif col == 'month':
                        # Scale month separately if not in scaler
                        max_month_value = 300  # 25 years * 12 months as a reasonable range
                        processed_row[col] = processed_row[col] / max_month_value
                    else:
                        processed_row[col] = 0
                except Exception as e2:
                    print(f"Manual scaling failed for {col}: {e2}")
                    processed_row[col] = 0

    # Step 4: Prepare the input for the model
    # Remove the target variable if it exists
    if 'resale_price' in processed_row.columns:
        x = processed_row.drop('resale_price', axis=1)
    else:
        x = processed_row

    # Convert to numpy array
    x = x.values.astype(np.float32)

    # Reshape for LSTM: [batch_size, sequence_length, num_features]
    # Original model expects each feature as its own sequence element with dim 1
    x_tensor = torch.tensor(x, dtype=torch.float32).view(1, -1, 1).to(device)

    # Step 5: Make prediction
    model.eval()
    with torch.no_grad():
        h_0 = torch.zeros(model.num_layers, 1, model.hidden_size).to(device)
        c_0 = torch.zeros(model.num_layers, 1, model.hidden_size).to(device)

        try:
            # This matches the original model's expected input shape
            _, _, output = model(x_tensor, h_0, c_0)
            final_output = output[:, -1, :].cpu().numpy()[0][0]
        except Exception as e:
            print(f"Error during model prediction: {e}")
            # Try a different approach if the model has a different interface
            try:
                output = model(x_tensor, h_0, c_0)
                if isinstance(output, tuple):
                    final_output = output[0].cpu().numpy()[0][0]
                else:
                    final_output = output.cpu().numpy()[0][0]
            except Exception as e2:
                print(f"Second attempt at prediction failed: {e2}")
                return float('nan')

    # Step 6: Descale the prediction
    try:
        if hasattr(scaler, 'feature_names_in_'):
            price_idx = list(scaler.feature_names_in_).index('resale_price')

            # Get the min and max values for resale_price
            min_val = scaler.data_min_[price_idx]
            max_val = scaler.data_max_[price_idx]

            # Reverse the min-max scaling
            predicted_price = final_output * (max_val - min_val) + min_val
            return predicted_price
        else:
            # Try a generic approach
            print("Warning: Scaler doesn't have feature_names_in_ attribute.")
            return final_output * 1000000  # Rough descaling assuming normalized 0-1 range
    except (ValueError, AttributeError) as e:
        print(f"Warning: Could not properly descale the prediction. Error: {e}")
        # Try finding it in continuous_cols
        if 'resale_price' in continuous_cols:
            price_idx = continuous_cols.index('resale_price')
            # Make a guess at the min/max values
            try:
                min_val = scaler.data_min_[price_idx]
                max_val = scaler.data_max_[price_idx]
                predicted_price = final_output * (max_val - min_val) + min_val
                return predicted_price
            except:
                print("Warning: Returning raw prediction value.")
                return final_output
        else:
            print("Warning: Returning raw prediction value.")
            return final_output

In [46]:
sample_rows = [
    {
        "month": "2025-01",
        "town": "GEYLANG",
        "flat_type": "3 ROOM",
        "block": "70",
        "street_name": "CIRCUIT RD",
        "storey_range": "01 TO 03",
        "floor_area_sqm": 56,
        "flat_model": "Standard",
        "lease_commence_date": 1969,
        "remaining_lease": "43 years 01 month",
        "resale_price": 358000
    },
    {
        "month": "2022-11",
        "town": "GEYLANG",
        "flat_type": "3 ROOM",
        "block": "57",
        "street_name": "CIRCUIT RD",
        "storey_range": "01 TO 03",
        "floor_area_sqm": 56,
        "flat_model": "Standard",
        "lease_commence_date": 1969,
        "remaining_lease": "43 years",
        "resale_price": 323000
    },
    {
        "month": "2025-02",
        "town": "GEYLANG",
        "flat_type": "3 ROOM",
        "block": "57",
        "street_name": "CIRCUIT RD",
        "storey_range": "01 TO 03",
        "floor_area_sqm": 56,
        "flat_model": "Standard",
        "lease_commence_date": 1969,
        "remaining_lease": "43 years",
        "resale_price": 325000
    },
    {
        "month": "2025-02",
        "town": "GEYLANG",
        "flat_type": "3 ROOM",
        "block": "93",
        "street_name": "JLN DUA",
        "storey_range": "01 TO 03",
        "floor_area_sqm": 62,
        "flat_model": "Standard",
        "lease_commence_date": 1969,
        "remaining_lease": "43 years",
        "resale_price": 340000
    },
    {
        "month": "2025-02",
        "town": "GEYLANG",
        "flat_type": "3 ROOM",
        "block": "95",
        "street_name": "OLD AIRPORT RD",
        "storey_range": "01 TO 03",
        "floor_area_sqm": 53,
        "flat_model": "Standard",
        "lease_commence_date": 1969,
        "remaining_lease": "43 years",
        "resale_price": 335000
    },
    {
        "month": "2025-02",
        "town": "PUNGGOL",
        "flat_type": "5 ROOM",
        "block": "670B",
        "street_name": "EDGEFIELD PLAINS",
        "storey_range": "16 TO 18",
        "floor_area_sqm": 113,
        "flat_model": "Premium Apartment",
        "lease_commence_date": 2016,
        "remaining_lease": "90 years 05 months",
        "resale_price": 895000
    },
    {
        "month": "2025-01",
        "town": "BUKIT TIMAH",
        "flat_type": "4 ROOM",
        "block": "4",
        "street_name": "TOH YI DR",
        "storey_range": "10 TO 12",
        "floor_area_sqm": 104,
        "flat_model": "Model A",
        "lease_commence_date": 1988,
        "remaining_lease": "62 years 09 months",
        "resale_price": 960000
    },
]

In [47]:
# Path to your models folder
model_folder = "/content/drive/My Drive/Deep Learning Project/Models/"

# Loop through each row and load the corresponding model
for row in sample_rows:
    # Generate model filename
    storey = row["storey_range"].replace(" ", "_")
    town = row["town"].replace(" ", "_")
    flat_type = row["flat_type"].replace(" ", "_")
    if device.type == "cpu":
      model_filename = f"{storey}_{town}_{flat_type}_bundle_CPU.pkl"
    else:
      model_filename = f"{storey}_{town}_{flat_type}_bundle_CUDA.pkl"

    model_path = os.path.join(model_folder, model_filename)

    try:
        # Load model bundle
        with open(model_path, 'rb') as f:
            bundle = pickle.load(f)

        # Create the model first
        model = bundle['model_class'](
            input_size=1, hidden_size=64, num_layers=4, output_size=1
        )

        # Then load the state dict with proper device handling
        if torch.cuda.is_available():
            model.load_state_dict(bundle['model_state_dict'])
            model = model.to(device)
        else:
            # Convert state dict tensors to CPU
            state_dict = {}
            for key, value in bundle['model_state_dict'].items():
                if isinstance(value, torch.Tensor):
                    state_dict[key] = value.cpu()
                else:
                    state_dict[key] = value

            model.load_state_dict(state_dict)
            model = model.to(device)

        model.eval()

        label_encoders = bundle['label_encoders']
        scaler = bundle['scaler']

        # Convert row to DataFrame and predict
        row_df = pd.DataFrame([row])
        predicted_price = predict_house_price(model, row_df, label_encoders, scaler, device)
        error_percent = abs(predicted_price - row["resale_price"]) / row["resale_price"] * 100
        print(f"{model_filename}: [predicted_price: {predicted_price:.2f}, actual_price: {row['resale_price']}, error%: {error_percent:.2f}%]")

    except FileNotFoundError:
        print(f"Model not found for: {model_filename}")
    except Exception as e:
        print(f"Error loading model for {model_filename}: {e}")

01_TO_03_GEYLANG_3_ROOM_bundle_CPU.pkl: [predicted_price: 322617.32, actual_price: 358000, error%: 9.88%]
Found new category in remaining_lease. Using default encoding.
New categories in remaining_lease (using default encoding): ['43 years']
01_TO_03_GEYLANG_3_ROOM_bundle_CPU.pkl: [predicted_price: 316459.03, actual_price: 323000, error%: 2.03%]
Found new category in remaining_lease. Using default encoding.
New categories in remaining_lease (using default encoding): ['43 years']
01_TO_03_GEYLANG_3_ROOM_bundle_CPU.pkl: [predicted_price: 316459.03, actual_price: 325000, error%: 2.63%]
Found new category in street_name. Using default encoding.
New categories in street_name (using default encoding): ['JLN DUA']
Found new category in remaining_lease. Using default encoding.
New categories in remaining_lease (using default encoding): ['43 years']
01_TO_03_GEYLANG_3_ROOM_bundle_CPU.pkl: [predicted_price: 318416.71, actual_price: 340000, error%: 6.35%]
Found new category in remaining_lease. Us